# EDA of the Abalone dataset

Goal: predict the age of an abalone from physical measurements instead of counting shell rings under a microscope.

The target is `Rings`; the age in years is `Rings + 1.5`, so predicting `Rings` is equivalent to predicting age.

# Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 500)
sns.set_theme()

DATA_PATH = Path("../data/abalone.csv")
TARGET = "Rings"

# Data

In [ ]:
df = pd.read_csv(DATA_PATH)
numeric_features = df.select_dtypes("number").columns.drop(TARGET).tolist()

print(df.shape)
df.head()

# Overview

Types, summary statistics, missing values and duplicates.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print(f"\nDuplicated rows: {df.duplicated().sum()}")

# Target: Rings

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df[TARGET], discrete=True, ax=ax)
ax.set_title("Distribution of Rings")
plt.show()

df[TARGET].describe()

# Categorical feature: Sex

`M` = male, `F` = female, `I` = infant.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Sex", ax=axes[0])
axes[0].set_title("Count by Sex")
sns.boxplot(data=df, x="Sex", y=TARGET, ax=axes[1])
axes[1].set_title("Rings by Sex")
plt.show()

# Numeric features

In [ ]:
df[numeric_features].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[numeric_features + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for feature, ax in zip(numeric_features, axes.flat):
    sns.scatterplot(data=df, x=feature, y=TARGET, hue="Sex", alpha=0.4, s=10, ax=ax)
    ax.set_title(f"{feature} vs Rings")
plt.tight_layout()
plt.show()

# Data quality checks

Physically impossible values: a height of 0, or a part (shucked / viscera / shell) heavier than the whole abalone.

In [ ]:
zero_height = df[df["Height"] == 0]
parts_heavier_than_whole = df[
    (df["Shucked weight"] > df["Whole weight"])
    | (df["Viscera weight"] > df["Whole weight"])
    | (df["Shell weight"] > df["Whole weight"])
]

print(f"Rows with zero height: {len(zero_height)}")
print(f"Rows with a part heavier than the whole: {len(parts_heavier_than_whole)}")
zero_height

In [ ]:
df.nlargest(5, "Height")

# Conclusions

- **Clean dataset**: ~4,177 rows, no missing values, no duplicates. Only `Sex` is categorical and needs encoding (e.g. one-hot).
- **Target**: `Rings` ranges from 1 to 29, centred around 9-10 and right-skewed; very old abalones are rare, so errors on them will be larger.
- **Sex**: the three classes are balanced. Infants (`I`) have clearly fewer rings, while males and females are very similar, so `Sex` is mostly useful as an infant / adult signal.
- **Numeric features**: all sizes and weights are strongly correlated with each other (multicollinearity), and moderately correlated with `Rings`. `Shell weight` has the highest correlation with the target.
- **Non-linear relationship**: rings grow quickly with size for young abalones, then flatten and spread out for adults, so a linear model is a reasonable baseline but will struggle on older abalones.
- **Data quality**: 2 rows have a zero height, 2 rows have extreme heights (1.13 and 0.515, versus 0.25 for the next tallest) and 5 rows have a part heavier than the whole. These are few and can be dropped before training.